<a href="https://colab.research.google.com/github/RPG-coder/newswise-bbc-kaggle/blob/main/NewsWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle

In [2]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

In [3]:
!kaggle datasets download hgultekin/bbcnewsarchive
!unzip bbcnewsarchive.zip

Dataset URL: https://www.kaggle.com/datasets/hgultekin/bbcnewsarchive
License(s): copyright-authors
bbcnewsarchive.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  bbcnewsarchive.zip
replace bbc-news-data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: bbc-news-data.csv       


In [4]:
import pandas as pd
df = pd.read_csv('bbc-news-data.csv', sep='\t', on_bad_lines='skip')

In [5]:
df.columns.tolist()

['category', 'filename', 'title', 'content']

In [6]:
df.head(2)

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...


In [7]:
df.shape

(2225, 4)

In [8]:
df['category'].value_counts()

,count
category,
sport,511
business,510
politics,417
tech,401
entertainment,386


In [9]:
!pip install langchain langchain-anthropic langchain-community langchain-huggingface faiss-gpu langgraph sentence-transformers

In [10]:
from langchain_core.documents import Document

doc = [
    Document(
        page_content=row['content'],
        metadata  = {
            'title': row['title'],
            'category': row['category']
        }
    )
    for _, row in df.iterrows()
]

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(doc, embedding)

/tmp/ipykernel_22981/323419157.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

history = []

prompt = ChatPromptTemplate([
    ('system', """You are a news research assistant.
Use ONLY the provided context to answer the question.
Do not use any external knowledge.
If you are not sure about the answer, just say that you don't know."""),
    MessagesPlaceholder(variable_name='history'),
    ('human', """Question: {question}
Context: {context}
Response:""")
])

In [14]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-sonnet-4-6")

In [15]:
from typing import TypedDict

class NewsState(TypedDict):
  question: str
  output: str


In [16]:
def ask_question(state: NewsState) -> NewsState:
  return {
      'question': input("Enter a Question: ")
  }

In [17]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

def answer_question(state: NewsState) -> NewsState:
  chain = {
      'question': RunnableLambda(lambda s: s['question']),
      'context': RunnableLambda(lambda s: s['question']) | retriever,
      'history': RunnableLambda(lambda s: s['history']),
  } | prompt | llm | StrOutputParser()
  result = chain.invoke({
      'question': state['question'],
      'history': history
  })
  print('Agent:', result)
  history.append(HumanMessage(content=state['question']))
  history.append(AIMessage(content=result))
  return {
      'output': result
  }

In [18]:
def question_route(state: NewsState) -> str:
  if state['question'] == '/exit':
    return END
  return 'answer'

In [19]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(NewsState)
builder.add_node('ask', ask_question)
builder.add_node('answer', answer_question)

builder.add_edge(START, 'ask')
builder.add_conditional_edges('ask', question_route)
builder.add_edge('answer', 'ask')

In [20]:
from langgraph.checkpoint.memory import MemorySaver

saver = MemorySaver()
config = {'configurable': {'thread_id': 'newswise-1'}}

In [21]:
graph = builder.compile(checkpointer=saver)

In [22]:
result = graph.invoke({
    'question': '',
    'output': '',
}, config=config)
print(result)

Enter a Question: What happened in football?
Agent: Based on the provided context, here are the football-related events mentioned:

1. **Manchester United vs Arsenal**: Sir Alex Ferguson called on the Football Association to punish Arsenal's Thierry Henry for allegedly deliberately hitting Gabriel Heinze on the head with his knee during United's win. Ferguson also criticized Arsenal's overall discipline, citing 24 fouls during the game. Referee Mike Riley deemed the incident accidental.

2. **Middlesbrough vs Graz AK (UEFA Cup)**: Middlesbrough won 2-1, booking their place in the last 16 of the UEFA Cup. However, the victory was marred by an injury to winger Stewart Downing, who was carried off on a stretcher with a hamstring problem. Middlesbrough will next face Sporting Lisbon for a place in the quarter-finals.

*Note: The first document refers to a **rugby** incident involving England's Josh Lewsey in the Six Nations, not football.*
Enter a Question: What about politics news?
Agent: